# Semantic experiment scores

Experiments are rows; metrics are columns. **Higher is better for every metric.**
Each column has its own colour scale: pale cells are lower, deeper teal cells are
higher. **Bold, outlined cells mark the highest score in that metric, including
all ties.** A column where all experiments tie highlights every entry equally.

Values are shown as fractions to four decimal places; colours and maxima use the
full stored precision. Colours compare experiments **within a metric**, not
between different metrics. Missing scores appear as an em dash.

Change `CSV_PATH` below to use another CSV, then **Run All**. This notebook reads
the CSV and does not change it or run any recommender models.


Rows ending `__context_tasks_v1` add all raw task statements to occupation description/context prose. No numerical occupation features or disk embedding inputs enter semantic scoring. See [experiment protocol](context_tasks_v1/README.md). All results are development/oracle; context here is descriptive job prose, not rated O*NET Work Context.

In [1]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display

# Relative paths work from this notebook's folder or the project root.
# An absolute path can be supplied for a different CSV.
CSV_PATH = Path("semantic_experiments_consolidated.csv")

def resolve_csv(path):
    if path.is_absolute():
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base / path, base / "reports" / "ranking" / path):
            if candidate.is_file():
                return candidate.resolve()
    raise FileNotFoundError(f"Cannot find {path}. Set CSV_PATH to the full CSV path.")

csv_path = resolve_csv(CSV_PATH)
experiments = pd.read_csv(csv_path)
if "experiment" not in experiments.columns:
    raise ValueError("CSV requires an 'experiment' column.")
if experiments.empty or experiments["experiment"].isna().any() or experiments["experiment"].duplicated().any():
    raise ValueError("CSV must contain nonempty, unique experiment names.")

preferred = ["hit_at_1", "hit_at_5", "hit_at_10", "mrr", "ndcg_at_5", "ndcg_at_10",
             "recall_at_5", "recall_at_10", "recall_at_20", "recall_at_50"]
detected = [name for name in experiments.columns
            if name == "mrr" or re.fullmatch(r"(?:hit|ndcg|recall)_at_\d+", name)]
metrics = [name for name in preferred if name in detected]
metrics += [name for name in detected if name not in metrics]
if not metrics:
    raise ValueError("No recognized metric columns found.")

scores = experiments.set_index("experiment")[metrics].apply(pd.to_numeric, errors="raise")
if ((scores < 0) | (scores > 1)).any().any():
    raise ValueError("Metric scores must be fractions between 0 and 1.")
labels = {name: name.replace("hit_at_", "Hit@").replace("ndcg_at_", "NDCG@")
          .replace("recall_at_", "Recall@").replace("mrr", "MRR") for name in metrics}
scores = scores.rename(columns=labels).rename_axis("Experiment")
print(f"Input: {csv_path.name} | {len(scores)} experiments | {len(scores.columns)} metrics")


Input: semantic_experiments_consolidated.csv | 14 experiments | 10 metrics


In [2]:
def metric_colours(column):
    """Normalize separately within each metric; preserve every maximum tie."""
    low, high = column.min(), column.max()
    styles = []
    for value in column:
        if pd.isna(value):
            styles.append("background-color: #f1f5f9; color: #64748b;")
            continue
        proportion = 1.0 if high == low else (value - low) / (high - low)
        pale, teal = (244, 248, 250), (94, 190, 171)
        rgb = tuple(round(a + proportion * (b - a)) for a, b in zip(pale, teal))
        style = f"background-color: rgb{rgb}; color: #102a2a;"
        if value == high:
            style += " font-weight: 700; box-shadow: inset 0 0 0 2px #147d64;"
        styles.append(style)
    return styles

styled_scores = (
    scores.style
    .apply(metric_colours, axis=0)
    .format("{:.4f}", na_rep="—")
    .set_caption("Experiment scores · colour scaled within each metric · outlined cells are column maxima")
    .set_table_styles([
        {"selector": "", "props": [("border-collapse", "separate"), ("border-spacing", "3px"),
                                     ("font-family", "Arial, sans-serif"), ("font-size", "13px")]},
        {"selector": "caption", "props": [("text-align", "left"), ("font-size", "15px"),
                                            ("font-weight", "600"), ("padding", "12px 0")]},
        {"selector": "th.col_heading", "props": [("background-color", "#183b44"), ("color", "white"),
                                                   ("padding", "12px"), ("text-align", "center")]},
        {"selector": "th.row_heading", "props": [("text-align", "left"), ("font-weight", "500"),
                                                   ("padding", "10px 14px"), ("white-space", "nowrap")]},
        {"selector": "td", "props": [("padding", "11px 14px"), ("text-align", "right"),
                                       ("font-variant-numeric", "tabular-nums")]},
    ])
)
display(styled_scores)


,Hit@1,Hit@5,Hit@10,MRR,NDCG@5,NDCG@10,Recall@5,Recall@10,Recall@20,Recall@50
Experiment,,,,,,,,,,
weighted_dot_product,0.0500,0.1667,0.3833,0.1415,0.1500,0.1958,0.0899,0.1674,0.3039,0.5898
minilm_top100,0.0250,0.0750,0.1417,0.0736,0.0751,0.0997,0.0475,0.1001,0.2039,0.4591
minilm_groups5,0.0167,0.1667,0.3833,0.1163,0.1378,0.1842,0.0899,0.1674,0.3039,0.5898
minilm_groups10,0.0333,0.2083,0.3833,0.1340,0.1431,0.1867,0.0805,0.1674,0.3039,0.5898
mpnet_groups10,0.0417,0.1750,0.3833,0.1331,0.1399,0.1874,0.0819,0.1674,0.3039,0.5898
mpnet20_then_minilm10,0.0167,0.1250,0.2500,0.1018,0.1228,0.1600,0.0797,0.1576,0.3039,0.5842
minilm20_then_mpnet10,0.0417,0.1750,0.3250,0.1272,0.1391,0.1752,0.0813,0.1567,0.3039,0.5905
minilm_mpnet_equal_weights_groups10,0.0333,0.2167,0.3833,0.1333,0.1451,0.1864,0.0818,0.1674,0.3039,0.5898
minilm_groups5__context_tasks_v1,0.0250,0.1667,0.3833,0.1209,0.1399,0.1862,0.0899,0.1674,0.3039,0.5898


**Reading the table:** bold outlines identify the best *observed* score,
not statistical significance. Equal scores at group boundaries may follow from
the reranking constraints. Rerun the notebook whenever the CSV changes.

Dependencies: `pandas`, `jinja2`, and an IPython/Jupyter kernel (available in the
project's ranking-diagnostics environment).


## Paired configuration comparisons

Each delta is treatment minus control; positive is better. Historical predictions were re-evaluated with the same ten metric functions and fingerprint-matching labels. These differences are descriptive, not significance claims. Fixed grouping forces some membership-cutoff ties. Added text and chunking are a joint treatment.

In [3]:
raw = experiments.set_index("experiment")[metrics].apply(pd.to_numeric)
suffix = "__context_tasks_v1"
new_names = [name for name in raw.index if name.endswith(suffix)]
if not new_names:
    raise ValueError("Context/task experiment rows are missing")
paired = pd.DataFrame({name.removesuffix(suffix): raw.loc[name] - raw.loc[name.removesuffix(suffix)]
                       for name in new_names}).T.rename(columns=labels)
display(paired.style.format("{:+.4f}").set_caption("Context + tasks minus matching description-only control"))
dot_deltas = raw.loc[new_names].subtract(raw.loc["weighted_dot_product"], axis="columns").rename(columns=labels)
display(dot_deltas.style.format("{:+.4f}").set_caption("Context + tasks minus weighted dot product"))


,Hit@1,Hit@5,Hit@10,MRR,NDCG@5,NDCG@10,Recall@5,Recall@10,Recall@20,Recall@50
minilm_groups5,+0.0083,+0.0000,+0.0000,+0.0046,+0.0021,+0.0019,+0.0000,+0.0000,+0.0000,+0.0000
minilm_groups10,+0.0000,+0.0083,+0.0000,-0.0042,+0.0026,-0.0012,+0.0024,+0.0000,+0.0000,+0.0000
mpnet_groups10,+0.0000,+0.0333,+0.0000,+0.0056,+0.0034,-0.0010,+0.0013,+0.0000,+0.0000,+0.0000
mpnet20_then_minilm10,+0.0083,+0.0250,+0.0417,+0.0084,-0.0057,-0.0068,-0.0019,-0.0109,+0.0000,-0.0079
minilm20_then_mpnet10,-0.0250,-0.0250,-0.0500,-0.0220,-0.0269,-0.0201,-0.0116,-0.0074,+0.0000,-0.0079
minilm_mpnet_equal_weights_groups10,-0.0083,-0.0167,+0.0000,-0.0050,-0.0076,-0.0021,+0.0009,+0.0000,+0.0000,+0.0000


,Hit@1,Hit@5,Hit@10,MRR,NDCG@5,NDCG@10,Recall@5,Recall@10,Recall@20,Recall@50
experiment,,,,,,,,,,
minilm_groups5__context_tasks_v1,-0.0250,+0.0000,+0.0000,-0.0206,-0.0101,-0.0096,+0.0000,+0.0000,+0.0000,+0.0000
minilm_groups10__context_tasks_v1,-0.0167,+0.0500,+0.0000,-0.0117,-0.0042,-0.0102,-0.0070,+0.0000,+0.0000,+0.0000
mpnet_groups10__context_tasks_v1,-0.0083,+0.0417,+0.0000,-0.0028,-0.0067,-0.0093,-0.0066,+0.0000,+0.0000,+0.0000
mpnet20_then_minilm10__context_tasks_v1,-0.0250,-0.0167,-0.0917,-0.0313,-0.0329,-0.0426,-0.0121,-0.0207,+0.0000,-0.0135
minilm20_then_mpnet10__context_tasks_v1,-0.0333,-0.0167,-0.1083,-0.0363,-0.0378,-0.0406,-0.0202,-0.0181,+0.0000,-0.0072
minilm_mpnet_equal_weights_groups10__context_tasks_v1,-0.0250,+0.0333,+0.0000,-0.0132,-0.0126,-0.0115,-0.0072,+0.0000,+0.0000,+0.0000
